# Create BBRF (NARSAD) Awards (RESEARCH PATTERN, Drupal field-scrape)

Brain & Behavior Research Foundation (BBRF) is a US mental-health
research funder (formerly known as NARSAD; founded 1987). It funds
neuroscience and psychiatric research through Young Investigator,
Independent Investigator, and Distinguished Investigator grants plus
several prize programs (Lieber, Pardes, Goldman-Rakic, Colvin, Ruane,
Maltz, Klerman/Freedman).

**Awarding body:** Brain and Behavior Research Foundation - F4320306147 (US, DOI 10.13039/100000874)

**Source:** Drupal 10 grantee directory at bbrfoundation.org/grantees, paginated via `?page=N` (0-indexed). The scraper enumerates ~280 pages at 20 grantees/page and parses each Drupal field-name-X block — same idiom as Damon Runyon (#73) but with an additional `field-collection-view-final` inner structure that lists multiple grants per scientist.

**Schema choices:**
- One row per (scientist, grant_type, grant_year) tuple. Multi-grant scientists (a Young Investigator who later got an Independent Investigator) ship as multiple rows.
- `funder_award_id` = `bbrf-{name-slug}-{grant-type-slug}-{grant-year}`.
- `funder_scheme` = grant_type from source (e.g. `Young Investigator`, `Independent Investigator`, `Distinguished Investigator`, prize names).
- `funding_type` derived via SQL CASE on grant_type — fellowship for Young Investigator (early-career), research for Independent/Distinguished, prize for the named prizes (Lieber/Pardes/Goldman-Rakic/etc.).
- `amount` / `currency` populated via SQL CASE on `grant_type` from BBRF's own program-page-verified amounts: Young Investigator USD 70,000 ($35K/yr × 2 yr); Independent Investigator USD 100,000 ($50K/yr × 2 yr); Distinguished Investigator USD 100,000 (1-year). Named-prize rows (Staglin Award, future Lieber/Pardes/etc.) ship NULL with §6.7-waiver since BBRF does not publish single-laureate amounts for those. Expected coverage ~99% (Staglin = 4 rows NULL out of 6,818).
- `lead_investigator.affiliation.country`: ISO-2 code mapped from the directory's verbatim country-name field. Most rows are US; the international cohort spans GB/CA/DE/FR/AU/NL/CH/IL/CN/JP/KR/IN/BR/etc.
- Postnominals (Ph.D., M.D., Sc.D., D.O., etc.) are stripped from name fields before split_name; given_name/family_name reflect the academic name without titles.
- `illness` (e.g. depression, schizophrenia, bipolar disorder) carries through as descriptive metadata in the SQL transform but is not a structured filter column on the awards table.

**Source:** https://bbrfoundation.org/grantees?page=N (0..~279)

**S3 location:** `s3a://openalex-ingest/awards/bbrf/bbrf_grantees.parquet`

**Method:** Drupal field-name-X HTML scrape with paginated views — distinct from Damon Runyon's per-scientist-page approach (BBRF's grantee directory renders all field values inline on the listing page itself, so we don't fetch a detail page per scientist).

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.bbrf_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/bbrf/bbrf_grantees.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.bbrf_raw;

## Step 1.5: Inspect raw + grant_type/country distribution

In [ ]:
%sql
DESCRIBE openalex.awards.bbrf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.bbrf_raw LIMIT 5;

In [ ]:
%sql
-- Grant_type distribution. Young Investigator dominates (~70-80%).
SELECT grant_type, COUNT(*) AS rows,
       COUNT(institution) AS has_inst,
       COUNT(country) AS has_country,
       COUNT(grant_year) AS has_year
FROM openalex.awards.bbrf_raw
GROUP BY grant_type
ORDER BY rows DESC;

In [ ]:
%sql
-- Country distribution.
SELECT country, COUNT(*) AS rows
FROM openalex.awards.bbrf_raw
GROUP BY country
ORDER BY rows DESC
LIMIT 15;

In [ ]:
%sql
-- Year range.
SELECT MIN(grant_year) AS min_year, MAX(grant_year) AS max_year, COUNT(DISTINCT grant_year) AS years
FROM openalex.awards.bbrf_raw
WHERE grant_year RLIKE '^[0-9]{4}$';

## Step 1.6: Fail-fast - verify BBRF funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306147;

## Step 2: Transform to award schema

`description` reports the grant program and illness focus when available. `funding_type` is derived via CASE on grant_type. `amount`/`currency` populated via CASE on `grant_type` using the program-level uniform amounts from BBRF's own program pages (YI USD 70,000, II USD 100,000, DI USD 100,000); named-prize rows ship NULL.

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.bbrf_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306147  -- Brain and Behavior Research Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT('BBRF ', COALESCE(n.grant_type, 'Grant'),
           CASE WHEN n.grant_year IS NOT NULL THEN CONCAT(' (', n.grant_year, ')') ELSE '' END,
           ' - ', n.name) AS display_name,
    CASE
        WHEN n.grant_type IS NOT NULL AND n.illness IS NOT NULL AND n.illness <> ''
            THEN CONCAT(n.grant_type, ' awarded for ', n.illness, '-focused research.')
        WHEN n.grant_type IS NOT NULL
            THEN CONCAT(n.grant_type, '.')
        ELSE NULL
    END AS description,
    f.funder_id,
    n.funder_award_id,
    CASE
        WHEN LOWER(n.grant_type) = 'young investigator' THEN 70000.0
        WHEN LOWER(n.grant_type) = 'independent investigator' THEN 100000.0
        WHEN LOWER(n.grant_type) = 'distinguished investigator' THEN 100000.0
        ELSE NULL
    END AS amount,
    CASE
        WHEN LOWER(n.grant_type) RLIKE '^(young|independent|distinguished) investigator$' THEN 'USD'
        ELSE NULL
    END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    CASE
        WHEN LOWER(n.grant_type) RLIKE 'young investigator|trailblazer|early.{0,5}career' THEN 'fellowship'
        WHEN LOWER(n.grant_type) RLIKE 'independent investigator|distinguished investigator|research' THEN 'research'
        WHEN LOWER(n.grant_type) RLIKE 'lieber|pardes|goldman|ruane|colvin|maltz|klerman|freedman|prize|award' THEN 'prize'
        ELSE 'research'
    END AS funding_type,
    n.grant_type AS funder_scheme,
    'bbrf_narsad' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.grant_year AS INT) AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN n.name IS NULL OR n.name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.institution AS name,
                n.country AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.bbrf_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 137

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'bbrf_narsad' AND priority = 137;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    137 AS priority  -- BBRF / NARSAD
FROM openalex.awards.bbrf_awards;

## Step 6: Verification

§6.7 amount-coverage check passes at ~99% (only the 4 Staglin Award rows ship NULL by program design; the 6,813 YI/II/DI rows populate via SQL CASE). All other completeness checks should be 95%+.

In [ ]:
%sql
SELECT COUNT(*) AS total_bbrf_rows FROM openalex.awards.bbrf_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.bbrf_awards;

In [ ]:
%sql
-- §6.3 Data completeness. pct_amount expected ~99% (Staglin Award rows ship NULL by program design).
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_pi,
    COUNT(start_year) AS has_start_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.bbrf_awards;

In [ ]:
%sql
-- §6.7 amount check. Expect ~99% USD; only Staglin Award (4 rows) and any future named-prize rows ship NULL by program design.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies
FROM openalex.awards.bbrf_awards;

In [ ]:
%sql
-- funding_type split derived from grant_type.
SELECT funding_type, COUNT(*) AS rows
FROM openalex.awards.bbrf_awards
GROUP BY funding_type
ORDER BY rows DESC;

In [ ]:
%sql
-- Top grant_type values.
SELECT funder_scheme AS grant_type, COUNT(*) AS rows
FROM openalex.awards.bbrf_awards
GROUP BY funder_scheme
ORDER BY rows DESC
LIMIT 15;

In [ ]:
%sql
-- Country split.
SELECT lead_investigator.affiliation.country AS country, COUNT(*) AS rows
FROM openalex.awards.bbrf_awards
GROUP BY country
ORDER BY rows DESC
LIMIT 10;

In [ ]:
%sql
-- Year range distribution.
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.bbrf_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC
LIMIT 20;

In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.bbrf_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Sample 10 rows for eyeball QA.
SELECT id, SUBSTRING(display_name, 1, 70) AS title,
       funder_scheme AS grant_type, funding_type, start_year,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 45) AS institution,
       lead_investigator.affiliation.country AS country
FROM openalex.awards.bbrf_awards
ORDER BY start_year DESC, family
LIMIT 10;